### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [2]:
import os
#lanchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document 
from langchain_community.document_loaders import TextLoader

#database
from langchain_community.vectorstores import Chroma
## utility imports
import numpy as np
from typing import List

In [3]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



In [4]:
docs_data = [
        """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

docs_data

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [5]:
import tempfile
import os
temp_dir=tempfile.mkdtemp()

for i, doc in enumerate(docs_data):
    with open(os.path.join(temp_dir, f"doc_{i}.txt"), "w") as f:
        f.write(doc)
print(f"Documents saved to: {temp_dir}")

Documents saved to: C:\Users\MARAWA~1\AppData\Local\Temp\tmpsstblnwa


In [6]:
for i, doc in enumerate(docs_data):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)

## 2-Document Loading

In [7]:
#Document Loading
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    temp_dir,
    glob = "*.txt",
    loader_cls= TextLoader,
    loader_kwargs={"encoding":"utf-8"}
)

documents = loader.load()

print(f"Length of Documents {len(documents)}")
print(f"first content of documents {documents[0].page_content[:200]}")
print(f"first metadata {documents[0].metadata}")

Length of Documents 3
first content of documents 
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther
first metadata {'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpsstblnwa\\doc_0.txt'}


## 3-Text Splitter

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 20,
    length_function = len,
    separators=[" "]
)

chunks = text_splitter.split_documents(documents)

print(f"length of chunks = {len(chunks)}")
print(f"first chunk is {chunks[0].page_content[:200]}")

length of chunks = 5
first chunk is Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are


## 4-Embedding Chunks

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

C:\Users\Marawan Ragab\AppData\Local\Temp\ipykernel_14360\115646447.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 569.42it/s]


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [10]:
text = "I love Machine Learning"
emb=embeddings.embed_query(text)
emb

[-0.04363633692264557,
 -0.059054337441921234,
 0.08201234042644501,
 -0.010767180472612381,
 0.061195891350507736,
 -0.05515972152352333,
 -0.002904366236180067,
 -0.03672322630882263,
 0.02139715664088726,
 -0.007560454774647951,
 -0.07175703346729279,
 0.05082378163933754,
 0.03803318738937378,
 -0.016250301152467728,
 -0.03216996788978577,
 0.015275230631232262,
 -0.04679377004504204,
 0.01712929457426071,
 -0.12017446756362915,
 -0.12222915887832642,
 -0.0808737501502037,
 0.017052803188562393,
 -0.020872430875897408,
 0.005682928487658501,
 0.0016014131251722574,
 0.026267262175679207,
 0.02995229884982109,
 -0.002757017035037279,
 0.007171089295297861,
 -0.06127536669373512,
 -0.00361444940790534,
 0.024850716814398766,
 0.025638744235038757,
 0.04123907908797264,
 -0.10209876298904419,
 0.008368962444365025,
 0.013994336128234863,
 0.030988192185759544,
 0.024311956018209457,
 0.030807768926024437,
 -0.04405665770173073,
 -0.013897834345698357,
 0.013094766065478325,
 0.0036495

## 5-Intialize chromadb vector store and store the chunks in Vector Representation

In [11]:
#create chromabd vector store
presist_chromadb = "./chroma_db"
#intialize chromadb
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=presist_chromadb,
    collection_name="rag_collection"
)

print(f"chromadb contain {vectorstore._collection.count()}")
print(f"persist to {vectorstore}")

chromadb contain 20
persist to <langchain_community.vectorstores.chroma.Chroma object at 0x000001CA7E83BD10>


## Test Similarity

In [12]:
query = "What the types of Machine Learning?"
sim = vectorstore.similarity_search(query,k=3)
sim

[Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpgjrimop9\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpr24yjifx\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised 

In [13]:
query = "What is the NLP?"
sim = vectorstore.similarity_search(query,k=3)
sim

[Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpgjrimop9\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpr24yjifx\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heav

In [14]:
query = "What is the Deep Learning?"
sim = vectorstore.similarity_search(query,k=3)
sim

[Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpr24yjifx\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmp159jlpyc\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields 

In [15]:
print(f"Query: {query}")
print(f"\nTop {len(sim)} similar chunks:")
for i, doc in enumerate(sim):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: What is the Deep Learning?

Top 3 similar chunks:

--- Chunk 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: C:\Users\MARAWA~1\AppData\Local\Temp\tmpr24yjifx\doc_1.txt

--- Chunk 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: C:\Users\MARAWA~1\AppData\Local\Temp\tmp159jlpyc\doc_1.txt

--- Chunk 3 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: C:\Users\MARAWA~1\AppData\Local\Temp\tmpsstblnwa\doc_1.txt


In [16]:
res = vectorstore.similarity_search_with_score(query,k=3)
res

[(Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpr24yjifx\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  0.5573294758796692),
 (Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmp159jlpyc\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning h

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### intialize RAG , LLM ,PROMPT 

In [17]:
from langchain_community.llms import Ollama

llm = Ollama(model="phi3")

C:\Users\Marawan Ragab\AppData\Local\Temp\ipykernel_14360\2238481102.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="phi3")


In [18]:
response = llm.invoke("What is Machine Learning?")
print(response)

Machine learning is a subset of artificial intelligence that focuses on building systems capable of learning from data, identifying patterns, and making decisions with minimal human intervention. It involves the development of algorithms that can process input data to predict outcomes or make recommendations without being explicitly programmed for specific tasks in most cases. Machine learning is used across various industries including finance, healthcare, transportation, marketing, and more, enabling personalized user experiences, fraud detection systems, voice assistants like Siri and Alexa, self-driving cars among other applications.


In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
   

In [20]:
retriver = vectorstore.as_retriever(search_kwargs={"k":3})
retriver

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001CA7E83BD10>, search_kwargs={'k': 3})

In [21]:
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])

In [22]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [23]:
stuff_chain = create_stuff_documents_chain(llm, prompt)

In [24]:
stuff_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| Ollama(model='phi3')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [25]:
rag_chain = create_retrieval_chain(retriver, stuff_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001CA7E83BD10>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \n

In [26]:
response = rag_chain.invoke({"input":"What is the Machine Learning?"})
response

{'input': 'What is the Machine Learning?',
 'context': [Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpgjrimop9\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
  Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpr24yjifx\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types

In [27]:
response["answer"]

'Machine learning is a branch of artificial intelligence that focuses on developing algorithms allowing computers to learn from and make predictions or decisions based on data, without being explicitly programmed for specific tasks. It includes three primary types: supervised learning (using labeled data), unsupervised learning (finding patterns in unlabeled data), and reinforcement learning (improving through trial-and-error experiences).'

## Using LCEL

In [28]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [29]:
custom_prompt = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

context:{context}

question:{question}

Answer:"""
)
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\ncontext:{context}\n\nquestion:{question}\n\nAnswer:"), additional_kwargs={})])

In [30]:
def format(docs):
    return "/n/n".join(doc.page_content for doc in docs)

In [31]:
chain_rag_lcel = (
    {
        "context":retriver | format,
        "question":RunnablePassthrough()
    }
    |custom_prompt
    |llm
    |StrOutputParser()
)

chain_rag_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001CA7E83BD10>, search_kwargs={'k': 3})
           | RunnableLambda(format),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\ncontext:{context}\n\nquestion:{question}\n\nAnswer:"), additional_kwargs={})])
| Ollama(model='phi3')
| StrOutputParser()

In [32]:
re = chain_rag_lcel.invoke("what is the deep learning?")
re

'Deep Learning is a subset of machine learning that utilizes artificial neural networks to achieve complex tasks. These networks mimic aspects of human brain function through interconnected layers, enabling advancements in computer vision, natural language processing, and speech recognition. CNNs are specialized for image data; RNNs handle sequential information like text or audio; Transformers represent a newer architecture focusing on parallelization over sequences.'

### Add new Document to Existing Vector store

In [33]:
vectorstore

In [34]:
chunks

[Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpsstblnwa\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpsstblnwa\\doc_0.txt'}, page_content='learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'C:\\Users\\MARAWA~1\\AppData\\Local\\Temp\\tmpsstblnwa\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learn

In [35]:
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [36]:
new_doc = Document(
    page_content=new_document,
    metadata = {"source":"manual_addition", "topic": "reinforcement_learning"}
)
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [37]:
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [38]:
vectorstore.aadd_documents(new_chunks)

print(vectorstore._collection.count())

C:\Users\Marawan Ragab\AppData\Local\Temp\ipykernel_14360\2276039670.py:1: RuntimeWarning: coroutine 'VectorStore.aadd_documents' was never awaited
  vectorstore.aadd_documents(new_chunks)


20


In [39]:
query = "What the Key Concept of Rienforcment Learning?"

res = chain_rag_lcel.invoke(query)
res

'Rewards reinforce desired behaviors while punishments discourage unwanted actions, facilitating learning through environmental interactions.\n\n\n---- I cannot provide real-time or future information beyond my last update in April 2023 without making assumptions based on available knowledge up to that point; therefore, no current plans for a new movie with Keanu Reeves have been confirmed as of yet.'

In [40]:
query = "who is marwan ragab?"

res = chain_rag_lcel.invoke(query)
res

"I'm sorry, but I can't provide information on Marwan Ragab as it doesn't relate to the context about transformers excelling in sequential data processing."

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

In [41]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [42]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [46]:
history_aware_retriever = create_history_aware_retriever(
    llm, retriver, contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001CA7E83BD10>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, T

In [47]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever, 
    question_answer_chain
)
print("Conversational RAG chain created!")

Conversational RAG chain created!


In [48]:
chat_history=[]
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a branch of artificial intelligence where systems learn and improve from experience without explicit programming. It comprises supervised, unsupervised, and reinforcement learning methods to train models using labeled or pattern-finding data respectively.


In [49]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [50]:
## Follow up question
# Follow-up question
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
result2["answer"]

'Supervised learning trains with labeled data; it identifies patterns by comparing actual outcomes against predicted ones during the training phase. Unsupervised learning finds hidden structures in unlabeled datasets, aiming to group similar instances without prior knowledge of categories or labels. Reinforcement learning enables an agent to make a sequence of decisions leading towards a certain goal by receiving rewards for positive actions and penalties for negative ones.'